# 10 · Event-driven CD: choosing which version production runs

An external system — a Lambda, EventBridge, a webhook relay — fires a pipeline when
something happens. Two requirements pull against each other:

1. The caller must **not know a version**. It fires by name; its code never changes.
2. You must **keep control** over which version it reaches, and be able to change that —
   including backwards — without touching the caller.

In Flyte 1 this was launch-plan activation: `fetch_active_launchplan()` resolved the name,
and `update_launch_plan(..., ACTIVE)` chose the version.

**Learning goals**

1. Meet both requirements in Flyte 2 with a version label and no extra machinery
2. Trace any run back to the exact code it executed
3. Know why `auto_version="latest"` only solves half the problem
4. Know when a trigger is worth the extra tooling

## 1. The answer: a version label

Flyte 2 has no active/default version — a task is always addressed by an exact version.
But **task versions are mutable**: re-deploying an existing version swaps its code in
place. That is enough to build the pointer yourself.

Deploy each release twice — once under an immutable tag that is a permanent record, once
under a label meaning "what production runs now":

```
version "r1"    <- immutable; never overwritten
version "r2"    <- immutable; never overwritten
version "prod"  <- overwritten each release; the caller pins this
```

The caller pins the label, forever:

```python
task = flyte.remote.Task.get("event_driven.ingest", version="prod")
run  = flyte.run(task, object_key=..., event_time=...)
```

Requirement 1: the caller names a label, never a version. Requirement 2: you decide what
`prod` contains. Both using only `flyte deploy`.

In [ ]:
# Release r1: an immutable tag, then the same code under the label.
!flyte deploy --version r1   scripts/event_driven_cd/pipeline.py env
!flyte deploy --version prod scripts/event_driven_cd/pipeline.py env

In [ ]:
# Fire it the way the Lambda will. Names a label; no version anywhere.
!python scripts/event_driven_cd/invoke.py --object-key s3://bucket/a.nc

In [ ]:
# Release r2. Production follows; the caller is untouched.
!flyte deploy --version r2   scripts/event_driven_cd/pipeline.py env
!flyte deploy --version prod scripts/event_driven_cd/pipeline.py env
!python scripts/event_driven_cd/invoke.py --object-key s3://bucket/b.nc

In [ ]:
# Roll back: check out the older commit, then re-publish under the label.
!flyte deploy --version prod scripts/event_driven_cd/pipeline.py env
!python scripts/event_driven_cd/invoke.py --object-key s3://bucket/c.nc

The caller's code did not change once across any of that.

One practical note: the label deploy re-uploads a bundle identical to the one the tag
deploy just sent. If your images have code baked in, `--copy-style none` makes it a
metadata-only write. Not baking is the default though — Flyte 2 ships code as a bundle to
blob storage and the pod pulls it at runtime, so the images used here carry only the
interpreter and dependencies, and `--copy-style none` would register a label with no code
to run.

## 2. Which code is a release running?

Runs launched through the label record their version as `prod`, so read the **code
bundle** instead. It is content-hashed, so matching it against your immutable tags
identifies the release exactly:

```python
def bundle(version):
    args = list(remote.Task.get("event_driven.ingest", version=version).fetch()
                .pb2.spec.task_template.container.args)
    return args[args.index("--tgz") + 1].rsplit("/", 1)[-1]

bundle("r2") == bundle("prod")   # True -> prod is running r2's code
```

Every run stores this too, so past runs stay traceable to exact code — useful when you
need to answer "which code produced this output". `invoke.py` prints it before launching.

Resist keeping a release marker in your source to track this. It is duplicated state that
drifts, and when it drifts it misreports what ran.

## 3. Why not `auto_version="latest"`

```python
task = flyte.remote.Task.get("event_driven.ingest", auto_version="latest")
```

This resolves to the most recently deployed version, so newest deploy always wins. The
caller stays version-ignorant, so requirement 1 is met.

Requirement 2 is not: there is nothing to pin, and the only way off a bad version is to
deploy another one. A good default in development, the wrong one for an event-driven
production path.

## 4. When a trigger is worth it

A **trigger** is a named pointer that stores which version it designates, so you move it
without redeploying. Two things it gives you that the label does not:

- **Rollback without rebuilding.** One write, so it still works when the old code no
  longer builds — and it cannot overwrite the tag you are rolling back to. With a label,
  rolling back means redeploying from an old commit; if your working tree isn't exactly
  that commit, you overwrite the record you were restoring.
- **Release history.** Every move is a recorded revision with an author, rather than only
  "who deployed last".

The costs: no CLI or console support for moving a pointer, so it takes a small script; and
a trigger currently requires a schedule, so the pointer declares an inert cron and is
deployed inactive.

In [ ]:
# The trigger variant, for comparison.
!flyte deploy --version r1 scripts/event_driven_cd/pointer_pipeline.py env
!flyte deploy --version r2 scripts/event_driven_cd/pointer_pipeline.py env
!python scripts/event_driven_cd/repoint.py prod event_driven_pointer.ingest --to r1
!python scripts/event_driven_cd/invoke.py --object-key s3://bucket/d.nc --mode pointer

## 5. Deploying publishes

Both approaches share one property worth planning around. In Flyte 1, registering a
version was inert and activation was the gate. In Flyte 2 there is no inert
"registered but not yet exposed" state: a deploy reaches name-only callers immediately,
and re-points a task's triggers.

So rollback is your safety mechanism, not a staging gate. If you need to verify before
production sees a release, deploy to a non-production domain first and promote by
deploying to prod.

## Recap

| | caller version-ignorant | you control the version | rollback | release history |
|---|---|---|---|---|
| `auto_version="latest"` | yes | **no** | redeploy | version list |
| **version label** | yes | yes | rebuild + redeploy | last deploy only |
| trigger pointer | yes | yes | one write, no rebuild | full revisions |

- Task versions are mutable, so a label is a pointer — deploy an immutable tag plus a label
- The caller pins the label and never changes across releases or rollbacks
- Code bundles are content-hashed, so runs stay traceable to exact code
- `auto_version="latest"` gives caller ignorance but takes away your control
- Reach for a trigger when rollback speed or release history justifies the tooling
- Deploying publishes; use domains if you need a staging gate